# Economy of Bits — Notebook 04: the mechanism

Notebook 03 gave one clean win and one honest failure.

**The win.** Every published JRFM claim survives the panel repair — α 1.467 → 1.141 at
*p* = 0.0062 against a published 1.491 → 1.141 at *p* = 0.006, with 2016 coverage up from
391 firms to 403. That table is ready for the editor.

**The failure.** Bit intensity does not explain the tail as H1 predicted, and the reason is
diagnosable. The quartile tail indices came back non-monotone (1.26, 1.86, 1.12, 1.90)
because the top decile of a middle quartile is not a tail of anything — Hill on a slice
estimates noise. The index ranks Incyte, AbbVie and Regeneron at the top and puts **Amazon
in the bottom quartile**, Microsoft in Q2, Apple and Alphabet in Q3.

But it *does* separate the top 5%, 10% and 25% from the rest at *p* < 0.0001, and says
nothing at the top 1%. It predicts being near the top and is silent about who is at it.

That pattern is the paper. This notebook tests why: the extreme tail is occupied by firms
whose accounting no longer looks bit-like, because serving AI consumes compute and compute
is physical capital. Microsoft's asset lightness fell 0.905 → 0.608 over the decade.

Whether that is general or one firm cannot be answered yet — Amazon, Meta and Alphabet are
missing PP&E in four to six years of ten, and they are exactly the firms the claim rests on.
Cell 3 fixes that first.

Upload `bits_inputs_v4.zip`. Runtime about 20 minutes, most of it cell 3.

## 1 · Setup

In [ ]:
# =============================================================================
# CELL 1 - Setup
#
# Notebook 03 delivered one clean win and one honest failure.
#
# WIN. Every published JRFM claim survives the panel repair. alpha 1.467 ->
# 1.141 at p = 0.0062 against a published 1.491 -> 1.141 at p = 0.006, with 2016
# coverage up from 391 firms to 403. That table goes to the editor as it stands.
#
# FAILURE. Bit intensity does not explain the tail the way H1 predicted, and the
# reason is diagnosable rather than mysterious:
#
#   * The quartile tail indices came out non-monotone and erratic -- 1.26, 1.86,
#     1.12, 1.90 -- because the "tail" of a bit-intensity quartile is not the
#     tail of anything. Hill on a slice of the middle of a distribution
#     estimates noise. That test design was wrong and is retired here.
#   * The index ranks INCY, ABBV, REGN, GILD and PFE at the top and puts Amazon
#     in the BOTTOM quartile, Microsoft in Q2, Apple and Alphabet in Q3. Gross
#     margin and asset lightness pick out pharmaceuticals and franchising, not
#     the economics of bits.
#   * But bit intensity DOES separate the top 5%, 10% and 25% of the index from
#     the rest at p < 0.0001, and says nothing at the top 1% (diff 0.02,
#     p = 0.74). It predicts being near the top and is silent about who is at it.
#
# That last pattern is the paper. The extreme tail -- the part the Pareto index
# measures -- is occupied by firms whose accounting no longer looks bit-like,
# because serving AI consumes compute and compute is physical capital. Microsoft's
# asset lightness falls from 0.905 to 0.608 across the decade.
#
# Whether that is a general pivot or one firm cannot be answered yet: Amazon,
# Meta and Alphabet are missing PP&E in 4 to 6 of ten years, and they are exactly
# the firms the claim depends on. Filling that gap is cell 3 and it comes first.
# =============================================================================
import os, re, sys, json, time, glob, zipfile, pathlib, datetime, shutil, warnings
import numpy as np, pandas as pd, requests

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 160)

RUN_TAG = "bits_04_mechanism"
BASE = pathlib.Path("/content") if pathlib.Path("/content").exists() else pathlib.Path.cwd()
ROOT = BASE / f"economy_of_bits_{RUN_TAG}"
OUTD, CACHE, FIGS = ROOT / "outputs", ROOT / "cache", ROOT / "outputs" / "figures"
for d in (OUTD, CACHE, FIGS):
    d.mkdir(parents=True, exist_ok=True)
saved_tables = []

def savetable(df, name):
    if df is None or not len(df):
        print(f"  skipped {name} (empty)"); return
    df.to_parquet(OUTD / f"{name}.parquet"); df.to_csv(OUTD / f"{name}.csv", index=False)
    saved_tables.append(name); print(f"  saved {name}  ({len(df):,} rows)")

def cached(name, builder, refresh=False):
    p = CACHE / f"{name}.parquet"
    if p.exists() and not refresh:
        d = pd.read_parquet(p); print(f"  cache hit: {name} ({len(d):,} rows)"); return d
    d = builder()
    if d is not None and len(d):
        d.to_parquet(p); print(f"  cached {name} ({len(d):,} rows)")
    return d

SEC_HEADERS = {"User-Agent": "Boston University Metropolitan College research "
                             "(spattna1@bu.edu)",
               "Accept-Encoding": "gzip, deflate"}

def sec_get(url, tries=3, timeout=60):
    for a in range(tries):
        try:
            time.sleep(0.15)
            r = requests.get(url, headers=SEC_HEADERS, timeout=timeout)
            if r.status_code == 200:
                return r.json()
            if r.status_code == 404:
                return None
            if r.status_code in (403, 429):
                time.sleep(5 * (a + 1)); continue
            return None
        except Exception:
            if a == tries - 1:
                return None
            time.sleep(2 * (a + 1))
    return None

PAPER_YEARS = list(range(2016, 2026))
print(f"writing to: {ROOT}")

## 2 · Inputs

In [ ]:
# =============================================================================
# CELL 2 - Inputs
#
# Upload bits_inputs_v4.zip: notebook 03's final panel and index, plus the EDGAR
# facts, so nothing is rebuilt from scratch.
# =============================================================================
def find_inputs():
    for z in sorted(glob.glob(str(BASE / "**" / "bits_inputs_v4.zip"), recursive=True)):
        dest = CACHE / "inputs"; dest.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)
        print(f"  extracted {pathlib.Path(z).name}"); return dest
    for d in (BASE / "inputs4", BASE, CACHE / "inputs"):
        if (pathlib.Path(d) / "panel_final.csv.gz").exists():
            return pathlib.Path(d)
    return None

SRC = find_inputs()
if SRC is None:
    raise FileNotFoundError("bits_inputs_v4.zip not found.")

rep = pd.read_csv(SRC / "panel_final.csv.gz"); rep["year"] = rep.year.astype(int)
facts = pd.read_csv(SRC / "edgar_facts_resolved.csv.gz")
tick2cik = pd.read_csv(SRC / "ticker_to_cik.csv")
coding = pd.read_csv(SRC / "platform_coding_final.csv")
pub_ineq = pd.read_csv(SRC / "inequality_by_year.csv")
print(f"\npanel  {len(rep):,} firm-years, {rep.ticker.nunique()} firms")
print(f"facts  {len(facts):,} observations")

CIK = dict(zip(tick2cik.ticker, tick2cik.cik))
rep["cik"] = rep.ticker.map(CIK)
rep["cap_share"] = rep.mktcap / rep.groupby("year").mktcap.transform("sum")
print(f"CIK matched for {100*rep.cik.notna().mean():.1f}% of rows")

## 3 · Fill the gaps where the claim lives

Full companyfacts for the 120 largest firms and anyone missing a component, plus capital
expenditure — which notebook 01 never collected and which measures the compute build-out
directly instead of by inference.

In [ ]:
# =============================================================================
# CELL 3 - Fill the gaps where the claim lives
#
# Notebook 01 pulled fundamentals from EDGAR's `frames` endpoint first, which is
# fast because one call returns every filer. But frames only carry filers whose
# fiscal period matches the calendar frame, and the companyfacts fallback ran
# only for firms missing revenue or assets. A firm with those two but missing
# PP&E never got the fallback.
#
# The cost of that shortcut lands exactly where it hurts most. Amazon is missing
# PP&E in six years of ten, Meta in six, Alphabet in five. Those three plus
# Microsoft are the firms any claim about capital intensity in the digital
# economy has to rest on.
#
# So: run companyfacts properly for the largest firms and for anyone missing a
# component, and add capital expenditure, which notebook 01 never collected and
# which measures the compute build-out directly rather than by inference.
# =============================================================================
CONCEPTS = {
    "revenue": ["RevenueFromContractWithCustomerExcludingAssessedTax",
                "RevenueFromContractWithCustomerIncludingAssessedTax",
                "Revenues", "SalesRevenueNet", "SalesRevenueGoodsNet"],
    "cogs": ["CostOfGoodsAndServicesSold", "CostOfRevenue", "CostOfGoodsSold",
             "CostOfServices"],
    "gross_profit": ["GrossProfit"],
    "rnd": ["ResearchAndDevelopmentExpense"],
    "sgna": ["SellingGeneralAndAdministrativeExpense"],
    "operating_income": ["OperatingIncomeLoss"],
    "ppe_net": ["PropertyPlantAndEquipmentNet",
                # REITs and some utilities use these instead, which is why real
                # estate showed only 43% PP&E coverage in notebook 01.
                "RealEstateInvestmentPropertyNet",
                "PropertyPlantAndEquipmentAndFinanceLeaseRightOfUseAssetAfterAccumulatedDepreciationAndAmortization"],
    "assets": ["Assets"],
    "goodwill": ["Goodwill"],
    "intangibles": ["IntangibleAssetsNetExcludingGoodwill", "FiniteLivedIntangibleAssetsNet"],
    # NEW: the compute build-out, measured rather than inferred.
    "capex": ["PaymentsToAcquirePropertyPlantAndEquipment",
              "PaymentsToAcquireProductiveAssets"],
    "depreciation": ["DepreciationDepletionAndAmortization",
                     "DepreciationAmortizationAndAccretionNet"],
}
STOCKS = {"ppe_net", "assets", "goodwill", "intangibles"}

def assign_fy(end, months=12):
    e = pd.Timestamp(end)
    return int((e - pd.Timedelta(days=int(30.4 * months / 2))).year)

def parse_facts(js, cik):
    out = []
    gaap = (js or {}).get("facts", {}).get("us-gaap", {})
    for field, tags in CONCEPTS.items():
        is_stock = field in STOCKS
        for rank, tag in enumerate(tags):
            node = gaap.get(tag)
            if not node:
                continue
            for unit, obs in node.get("units", {}).items():
                if unit != "USD":
                    continue
                for o in obs:
                    if o.get("form") not in ("10-K", "10-K/A", "20-F", "40-F"):
                        continue
                    if o.get("val") is None or not o.get("end"):
                        continue
                    if is_stock:
                        year = pd.Timestamp(o["end"]).year
                    else:
                        if not o.get("start"):
                            continue
                        days = (pd.Timestamp(o["end"]) - pd.Timestamp(o["start"])).days
                        if not (330 <= days <= 400):
                            continue
                        year = assign_fy(o["end"])
                    out.append({"cik": int(cik), "year": year, "field": field,
                                "tag": tag, "tag_rank": rank, "value": float(o["val"]),
                                "end": o["end"], "accn": o.get("accn"),
                                "source": "companyfacts"})
    return out

# Who needs it: the 120 largest firms by peak market cap, plus anyone missing a
# core component in any year. The size cut is what guarantees the tail is
# complete, which is the part the paper argues about.
have = facts.pivot_table(index=["cik", "year"], columns="field",
                         values="value", aggfunc="first").reset_index()
CORE = ["revenue", "assets", "ppe_net", "cogs"]
for c in CORE:
    if c not in have.columns:
        have[c] = np.nan
gap_ciks = set(have.loc[have[CORE].isna().any(axis=1), "cik"].dropna().astype(int))
big = (rep.groupby("cik").mktcap.max().sort_values(ascending=False)
       .head(120).index.dropna().astype(int))
NEED = sorted(set(big) | gap_ciks)
print(f"{len(NEED)} CIKs need the full companyfacts pass "
      f"({len(set(big))} largest, {len(gap_ciks)} with a gap)")
print(f"estimated {len(NEED)*0.5/60:.0f} minutes")

def fetch(ciks):
    rows, miss = [], 0
    for i, c in enumerate(ciks, 1):
        js = sec_get(f"https://data.sec.gov/api/xbrl/companyfacts/CIK{int(c):010d}.json")
        if js is None:
            miss += 1
        else:
            rows.extend(parse_facts(js, c))
        if i % 50 == 0:
            print(f"    {i}/{len(ciks)}, {len(rows):,} facts, {miss} unavailable")
    return pd.DataFrame(rows)

new_facts = cached("companyfacts_fill", lambda: fetch(NEED))
print(f"\n{len(new_facts):,} facts for {new_facts.cik.nunique() if len(new_facts) else 0} CIKs")

old = facts.copy()
old["tag_rank"] = [CONCEPTS.get(f, [t]).index(t) if t in CONCEPTS.get(f, []) else 99
                   for f, t in zip(old.field, old.tag)]
old["end"] = pd.NaT; old["accn"] = ""
allf = pd.concat([old, new_facts], ignore_index=True) if len(new_facts) else old
allf["src_rank"] = (allf.source == "frames").astype(int)
allf["end"] = pd.to_datetime(allf.get("end"), errors="coerce")
resolved = (allf.sort_values(["cik", "year", "field", "tag_rank", "src_rank", "end", "accn"],
                             ascending=[True, True, True, True, True, False, False])
            .groupby(["cik", "year", "field"], as_index=False)
            .first()[["cik", "year", "field", "value", "tag", "source"]])
print(f"{len(facts):,} -> {len(resolved):,} resolved firm-year-fields")
savetable(resolved, "edgar_facts_v2")

## 4 · Rebuild the components

In [ ]:
# =============================================================================
# CELL 4 - Rebuild the components, and check the gap actually closed
# =============================================================================
wide = resolved.pivot_table(index=["cik", "year"], columns="field",
                            values="value", aggfunc="first").reset_index()
wide.columns.name = None
F = rep.merge(wide, on=["cik", "year"], how="left")

# A field absent from EVERY filer never becomes a column, and the next line
# would then raise AttributeError rather than produce an empty component.
for c in CONCEPTS:
    if c not in F.columns:
        F[c] = np.nan
        print(f"  NOTE: no filer reported {c}; the component is empty")

F["gross_profit"] = F.gross_profit.fillna(F.revenue - F.cogs)
F["asset_light"] = 1 - (F.ppe_net / F.assets)
F["intangible_share"] = F.intangibles / F.assets          # excludes goodwill
F["goodwill_share"] = F.goodwill / F.assets
F["rnd_intensity"] = F.rnd / F.revenue
F["gross_margin"] = F.gross_profit / F.revenue
F["capex_intensity"] = F.capex / F.revenue
F["ppe_share"] = F.ppe_net / F.assets

COMPONENTS = ["asset_light", "intangible_share", "rnd_intensity", "gross_margin"]
EXTRAS = ["goodwill_share", "capex_intensity", "ppe_share"]
for c in COMPONENTS + EXTRAS:
    lo = F.groupby("year")[c].transform(lambda s: s.quantile(0.01))
    hi = F.groupby("year")[c].transform(lambda s: s.quantile(0.99))
    F[c] = F[c].clip(lo, hi)

print("coverage, notebook 03 vs now:")
PREV = {"asset_light": 84.0, "intangible_share": 92.2, "rnd_intensity": 37.8,
        "gross_margin": 63.2}
for c in COMPONENTS:
    now = 100 * F[c].notna().mean()
    print(f"  {c:<18}{PREV.get(c, float('nan')):5.1f}%  ->  {now:5.1f}%")
for c in EXTRAS:
    print(f"  {c:<18}    --   ->  {100*F[c].notna().mean():5.1f}%")

GIANTS = ["AAPL", "MSFT", "GOOGL", "GOOG", "AMZN", "META", "NVDA", "AVGO", "TSLA", "LLY"]
g = F[F.ticker.isin(GIANTS)]
print(f"\nthe firms the capital-intensity claim rests on: PP&E present in "
      f"{int(g.asset_light.notna().sum())} of {len(g)} firm-years "
      f"(was 57 of 70 for the first eight)")
print("\nasset lightness, by firm and year:")
display(g.pivot_table(index="year", columns="ticker", values="asset_light").round(3))
print("\ncapital expenditure as a share of revenue:")
display(g.pivot_table(index="year", columns="ticker", values="capex_intensity").round(3))
savetable(F[["year", "ticker", "cik", "sector", "mktcap", "cap_share"]
            + COMPONENTS + EXTRAS], "components_v2")

## 5 · Is the digital economy still asset-light?

The central question, and the one that reframes everything after it.

In [ ]:
# =============================================================================
# CELL 5 - THE CENTRAL QUESTION: is the digital economy still asset-light?
#
# Machine, Platform, Crowd chapter 6 rests on information goods being free to
# reproduce and instant to deliver. If that were still the operative economics
# of the firms at the top of the index, their balance sheets would be getting
# LIGHTER as they grow. The JRFM paper's Future Scope section predicted the
# opposite: that AI moves the binding constraint from software, which is
# non-rival, to compute, which is rival and physical.
#
# This is now a measurable claim rather than a speculation, and it is the first
# thing worth testing because the answer reframes everything after it.
#
# Three groups, fixed in advance:
#   TAIL      the firms in the top 1% of index value in any year -- the ones the
#             Pareto index is actually estimated from
#   DIGITAL   GICS digital, excluding the tail, so the two do not overlap
#   REST      everything else
# =============================================================================
import statsmodels.api as sm
from scipy import stats as st

DIGITAL_SECTORS = {"Information Technology", "Communication Services"}
DIGITAL_EXTRA = {"AMZN", "TSLA", "NFLX", "BKNG", "EBAY", "ABNB", "UBER", "DASH", "SHOP"}
F["gics_digital"] = (F.sector.isin(DIGITAL_SECTORS)
                     | F.ticker.str.upper().isin(DIGITAL_EXTRA))
# A firm is "tail" if it EVER reaches the top 1%, so the group does not change
# composition year to year and the trend is not an artefact of entry and exit.
top1 = F.groupby("year").cap_share.transform(lambda s: s >= s.quantile(0.99))
TAIL_FIRMS = set(F.loc[top1, "ticker"].unique())
F["group3"] = np.where(F.ticker.isin(TAIL_FIRMS), "tail",
                       np.where(F.gics_digital, "digital", "rest"))
print("firms per group:")
print(F.groupby("group3").ticker.nunique().to_string())
print(f"\nthe tail group: {sorted(TAIL_FIRMS)}")

for var, label in [("asset_light", "asset lightness  (1 - PP&E/assets)"),
                   ("capex_intensity", "capital expenditure / revenue"),
                   ("gross_margin", "gross margin")]:
    piv = F.pivot_table(index="year", columns="group3", values=var)
    print(f"\n{label}:")
    display(piv.round(4))
    rows = []
    for grp in piv.columns:
        s = piv[grp].dropna()
        if len(s) < 5:
            continue
        X = sm.add_constant(s.index.values - np.mean(s.index.values))
        fit = sm.OLS(s.values, X).fit(cov_type="HAC", cov_kwds={"maxlags": 3})
        rows.append({"variable": var, "group": grp,
                     "first": round(float(s.iloc[0]), 4),
                     "last": round(float(s.iloc[-1]), 4),
                     "slope_per_year": round(float(fit.params[1]), 5),
                     "p_hac": round(float(fit.pvalues[1]), 4)})
    t = pd.DataFrame(rows)
    display(t)
    savetable(t, f"trend_{var}_by_group")

print("\n" + "=" * 72)
print("READ THIS CAREFULLY")
print("=" * 72)
print("If the tail group's asset lightness is FALLING while the rest of the")
print("index is flat or rising, the firms at the top of the distribution are")
print("becoming more capital-intensive than the average firm -- which is the")
print("opposite of what an economy of weightless goods would produce, and is")
print("the Future Scope prediction confirmed.")
print()
print("If instead the fall is confined to one or two firms, say so. Notebook 03")
print("appeared to show a pivot that was almost entirely Microsoft: its asset")
print("lightness went 0.905 -> 0.608 while Nvidia went 0.947 -> 0.950, because")
print("Nvidia is fabless and the fabs belong to TSMC. A firm-by-firm table is")
print("printed above precisely so that a group average cannot hide that.")

## 6 · Two index specifications, side by side

In [ ]:
# =============================================================================
# CELL 6 - Rebuild the index, and be honest about what it ranks
#
# Two specifications, reported side by side rather than one chosen quietly:
#
#   BI-A  the notebook 03 index: asset lightness, intangible share, R&D
#         intensity, gross margin. Kept unchanged so the comparison is clean.
#   BI-B  drops asset lightness. The reason is cell 5: if the most bit-intensive
#         firms are now the most capital-intensive, then asset lightness has the
#         WRONG SIGN for the mechanism it was chosen to proxy, and including it
#         pushes exactly the wrong firms to the top.
#
# BI-B is not a fix applied to make a result appear. It is a component being
# dropped because the data says it measures something other than what it was
# picked for, and both versions are carried through every test that follows.
# =============================================================================
MIN_COMPONENTS = 2
SPECS = {"BI_A": ["asset_light", "intangible_share", "rnd_intensity", "gross_margin"],
         "BI_B": ["intangible_share", "rnd_intensity", "gross_margin"]}
for c in set(sum(SPECS.values(), [])):
    F[f"z_{c}"] = F.groupby("year")[c].transform(
        lambda s: (s - s.mean()) / s.std() if s.std() and s.notna().sum() > 5 else np.nan)
for name, comps in SPECS.items():
    zc = [f"z_{c}" for c in comps]
    F[name] = F[zc].mean(axis=1)
    F.loc[F[comps].notna().sum(axis=1) < MIN_COMPONENTS, name] = np.nan
    F[name] = F.groupby("year")[name].transform(lambda s: (s - s.mean()) / s.std())
    print(f"{name}: {int(F[name].notna().sum()):,} firm-years "
          f"({100*F[name].notna().mean():.1f}%), components {comps}")
print(f"\ncorr(BI_A, BI_B) = {F[['BI_A','BI_B']].corr().iloc[0,1]:+.3f}")

for name in SPECS:
    d = F.dropna(subset=[name])
    print(f"\n--- {name} ---")
    print(f"corr with GICS digital: "
          f"{d[[name,'gics_digital']].astype(float).corr().iloc[0,1]:+.3f}")
    last = d[d.year == d.year.max()].sort_values(name, ascending=False)
    print("top 12 in 2025: " + ", ".join(last.head(12).ticker))
    print("bottom 8:       " + ", ".join(last.tail(8).ticker))
    gt = last[last.ticker.isin(GIANTS)][["ticker", name]].round(2)
    print("the firms that carry the tail: " +
          ", ".join(f"{r.ticker} {r[name]:+.2f}" for _, r in gt.iterrows()))

print("\nmean by sector, both specifications:")
display(F.groupby("sector")[["BI_A", "BI_B"]].mean().round(3)
        .sort_values("BI_B", ascending=False))
print("\nThe question to ask of this table is not whether it is tidy but whether")
print("the ordering is one the theory would predict. Health Care at the top is a")
print("genuine finding if it survives -- a pill costs almost nothing to")
print("reproduce and its value sits in IP -- but it is not what chapter 6 is")
print("about, and the paper has to say which it is measuring.")
savetable(F[["year", "ticker", "sector", "mktcap", "cap_share", "gics_digital",
             "group3", "BI_A", "BI_B"] + COMPONENTS + EXTRAS], "index_v2")

## 7 · H1, with an instrument that can detect what it claims

Membership of the top *k*%, and a Gabaix–Ibragimov rank regression that puts both groups in
one equation so the difference in tail exponent is a single coefficient.

In [ ]:
# =============================================================================
# CELL 7 - H1 restated, with a test that can actually detect what it claims
#
# Notebook 03 split firms into bit-intensity quartiles and estimated a Hill tail
# index inside each. That was the wrong instrument. The Hill estimator needs a
# TAIL; the top decile of a middle quartile is not a tail of anything, and the
# result was noise -- 1.26, 1.86, 1.12, 1.90 across four ordered groups.
#
# The question H1 actually asks is whether bit-intensive firms are found at the
# top of the distribution. That is a membership question and it has a direct
# test: compare bit intensity inside and outside the top k%, and let k vary.
# Notebook 03's data already showed the shape -- significant at 5%, 10% and 25%,
# nothing at 1% -- and this cell establishes it properly, on both index
# specifications, with year fixed effects and firm-clustered standard errors so
# that ten repeated observations of Microsoft do not count as ten firms.
# =============================================================================
rows = []
for spec in ["BI_A", "BI_B"]:
    d = F.dropna(subset=[spec]).copy()
    for q in (0.01, 0.02, 0.05, 0.10, 0.25):
        d["in_top"] = d.groupby("year").cap_share.transform(
            lambda s: s >= s.quantile(1 - q)).astype(float)
        X = pd.get_dummies(d.year, prefix="y", drop_first=True).astype(float)
        X[spec] = d[spec].values
        X = sm.add_constant(X)
        fit = sm.OLS(d.in_top.values, X).fit(
            cov_type="cluster", cov_kwds={"groups": d.ticker.values})
        rows.append({"spec": spec, "top_pct": f"{q:.0%}",
                     "n_top": int(d.in_top.sum()), "n": len(d),
                     "coef": round(float(fit.params[spec]), 4),
                     "se": round(float(fit.bse[spec]), 4),
                     "t": round(float(fit.tvalues[spec]), 2),
                     "p": round(float(fit.pvalues[spec]), 4),
                     "mean_BI_in_top": round(float(d.loc[d.in_top == 1, spec].mean()), 3),
                     "mean_BI_rest": round(float(d.loc[d.in_top == 0, spec].mean()), 3)})
h1 = pd.DataFrame(rows)
print("H1 -- does bit intensity predict membership of the top k% of index value?")
print("(linear probability model, year fixed effects, SEs clustered by firm)\n")
display(h1)
savetable(h1, "h1_tail_membership")

print("\nThe pattern to look for is a coefficient that is positive and")
print("significant at 10% and 25% and vanishes at 1%. That would say bit")
print("intensity sorts the upper half of the distribution and is silent about")
print("the extreme tail -- which is the part the Pareto index measures, and")
print("therefore the part the first paper's result is about.")

# --- Gabaix-Ibragimov, the right way to compare tail exponents ---------------
# Two Hill estimates on two subsamples cannot be compared without assuming the
# subsamples are each Pareto. The rank regression puts both groups in ONE
# equation with an interaction, so the difference in tail exponent is a single
# coefficient with a standard error:
#
#   log(rank - 1/2) = a + b*log(size) + c*high_BI + d*high_BI*log(size)
#
# The tail exponent is -b for the low group and -(b+d) for the high group, so d
# is the difference and its p-value is the test. Gabaix and Ibragimov (2011)
# show the -1/2 shift removes the small-sample bias in the naive version.
print("\n" + "=" * 72)
print("GABAIX-IBRAGIMOV RANK REGRESSION, tail exponent by bit intensity")
print("=" * 72)
rows = []
for spec in ["BI_A", "BI_B"]:
    for frac in (0.10, 0.20):
        d = F.dropna(subset=[spec, "cap_share"]).copy()
        d["high_BI"] = d.groupby("year")[spec].transform(
            lambda s: (s >= s.median()).astype(float))
        parts = []
        for (y, hb), g in d.groupby(["year", "high_BI"]):
            g = g.sort_values("cap_share", ascending=False).copy()
            k = max(15, int(frac * len(g)))
            g = g.head(k).copy()
            g["rank"] = np.arange(1, len(g) + 1)
            parts.append(g)
        dd = pd.concat(parts)
        dd["log_rank"] = np.log(dd["rank"] - 0.5)
        dd["log_size"] = np.log(dd.cap_share)
        X = pd.DataFrame({"log_size": dd.log_size, "high_BI": dd.high_BI,
                          "inter": dd.high_BI * dd.log_size})
        X = sm.add_constant(X)
        fit = sm.OLS(dd.log_rank.values, X.values.astype(float)).fit(
            cov_type="cluster", cov_kwds={"groups": dd.ticker.values})
        b, dcoef = float(fit.params[1]), float(fit.params[3])
        rows.append({"spec": spec, "tail_frac": f"{frac:.0%}", "n": len(dd),
                     "alpha_low_BI": round(-b, 4),
                     "alpha_high_BI": round(-(b + dcoef), 4),
                     "difference": round(-dcoef, 4),
                     "p_difference": round(float(fit.pvalues[3]), 4)})
gi = pd.DataFrame(rows)
display(gi)
print("\nH1 predicts alpha_high_BI < alpha_low_BI, so a NEGATIVE difference.")
print("This is one coefficient with one p-value, which is what the quartile")
print("comparison could not give.")
savetable(gi, "h1_gabaix_ibragimov")

## 8 · H3 — Gibrat, or increasing returns to size?

In [ ]:
# =============================================================================
# CELL 8 - H3: is growth independent of size, as Gibrat's law requires?
#
# This is the hypothesis that separates the two candidate mechanisms, and it is
# the one notebook 03 never reached.
#
#   Gibrat's law: growth is proportionate and independent of size. A heavy tail
#   arises from accumulated random growth. Nothing about the firm's economics
#   needs to be special.
#
#   Demand-side scale: a user's benefit rises with the number of other users, so
#   larger firms grow FASTER. The tail is generated, not accumulated.
#
# Test: regress log growth on log size, with year fixed effects and firm
# clusters. Gibrat predicts a coefficient of zero. A positive coefficient is
# increasing returns to size. Then interact with bit intensity and with the
# platform indicator to ask whether any departure from Gibrat is concentrated
# where the theory says it should be.
# =============================================================================
G = F.sort_values(["ticker", "year"]).copy()
G["lag_cap"] = G.groupby("ticker").mktcap.shift(1)
G["lag_year"] = G.groupby("ticker").year.shift(1)
G = G[(G.year - G.lag_year) == 1]
G["growth"] = np.log(G.mktcap / G.lag_cap)
G["log_size"] = np.log(G.lag_cap)
G = G.replace([np.inf, -np.inf], np.nan).dropna(subset=["growth", "log_size"])
print(f"{len(G):,} firm-year growth observations, {G.ticker.nunique()} firms")

pl = dict(zip(coding.ticker, coding.platform.astype(bool)))
G["platform"] = G.ticker.map(pl).fillna(False).astype(float)

def gib(df, extra=None, label=""):
    X = pd.get_dummies(df.year, prefix="y", drop_first=True).astype(float)
    X["log_size"] = df.log_size.values
    if extra:
        for k, v in extra.items():
            X[k] = v
    X = sm.add_constant(X)
    f = sm.OLS(df.growth.values, X.astype(float)).fit(
        cov_type="cluster", cov_kwds={"groups": df.ticker.values})
    out = {"model": label, "n": len(df),
           "log_size_coef": round(float(f.params["log_size"]), 5),
           "log_size_p": round(float(f.pvalues["log_size"]), 4)}
    for k in (extra or {}):
        out[f"{k}_coef"] = round(float(f.params[k]), 5)
        out[f"{k}_p"] = round(float(f.pvalues[k]), 4)
    return out

rows = [gib(G, None, "Gibrat baseline")]
for spec in ["BI_A", "BI_B"]:
    d = G.dropna(subset=[spec])
    rows.append(gib(d, {spec: d[spec].values,
                        f"{spec}_x_size": (d[spec] * d.log_size).values},
                    f"with {spec} and its interaction"))
rows.append(gib(G, {"platform": G.platform.values,
                    "platform_x_size": (G.platform * G.log_size).values},
                "with platform and its interaction"))
gibrat = pd.DataFrame(rows)
display(gibrat)
savetable(gibrat, "h3_gibrat")

print("\nGibrat's law predicts log_size_coef = 0. A significantly POSITIVE")
print("coefficient means large firms grew faster than small ones over this")
print("decade, which is increasing returns to scale and is what a network-effect")
print("mechanism produces. A significantly NEGATIVE one is mean reversion.")
print("\nThe variance test matters as much as the mean. Gibrat also requires the")
print("VARIANCE of growth to be independent of size:")
G["size_decile"] = G.groupby("year").log_size.transform(
    lambda s: pd.qcut(s, 10, labels=False, duplicates="drop"))
v = G.groupby("size_decile").growth.agg(["count", "mean", "std"]).round(4)
display(v)
X = sm.add_constant(v.index.values.astype(float))
fv = sm.OLS(v["std"].values, X).fit()
print(f"growth SD on size decile: slope {float(fv.params[1]):+.5f}, "
      f"p = {float(fv.pvalues[1]):.4f}")
print("A declining SD across deciles is the standard finding and is evidence")
print("AGAINST pure Gibrat; it is what Stanley et al. and Sutton document.")
savetable(v.reset_index(), "h3_growth_variance_by_size")

## 9 · H4 — does the tail churn?

In [ ]:
# =============================================================================
# CELL 9 - H4: does the tail hold still, or does it churn?
#
# Livermore found that 40% of the trusts formed between 1888 and 1905 had failed
# by the early 1930s. Caves found the average market share of firms dominant in
# 1905 and still alive in 1929 fell from 69% to 45%. Concentration under a
# general purpose technology has historically been a phase, not an end state.
#
# Ten years cannot settle whether this episode is different, but it can say
# whether the tail so far is the same firms or a rotating cast -- and that is
# the difference between a structural claim and a snapshot.
# =============================================================================
d = F.dropna(subset=["cap_share"]).copy()
d["decile"] = d.groupby("year").cap_share.transform(
    lambda s: pd.qcut(s, 10, labels=False, duplicates="drop"))

piv = d.pivot_table(index="ticker", columns="year", values="decile")
pairs = []
for y in PAPER_YEARS[:-1]:
    if y in piv.columns and y + 1 in piv.columns:
        p = piv[[y, y + 1]].dropna()
        pairs.append(pd.DataFrame({"from": p[y].values, "to": p[y + 1].values}))
# reset the index: the same ticker appears once per year pair, and crosstab
# cannot reindex an axis carrying duplicate labels.
tr = pd.concat(pairs, ignore_index=True)
M = (pd.crosstab(tr["from"], tr["to"], normalize="index"))
print("year-to-year decile transition matrix (rows sum to 1, 9 = largest):")
display(M.round(3))
# Shorrocks index: (n - trace) / (n - 1). Zero means perfect immobility, one
# means a firm's next position is independent of its current one.
n = M.shape[0]
shorrocks = (n - np.trace(M.values)) / (n - 1)
print(f"\nShorrocks mobility index: {shorrocks:.3f}")
print("  0 = nothing ever moves, 1 = position is uninformative about next year")
savetable(M.reset_index(), "h4_decile_transitions")

rows = []
for q, lab in [(0.01, "top 1%"), (0.05, "top 5%"), (0.10, "top 10%")]:
    mem = {}
    for y in PAPER_YEARS:
        s = d[d.year == y]
        if len(s):
            mem[y] = set(s.loc[s.cap_share >= s.cap_share.quantile(1 - q), "ticker"])
    ys = sorted(mem)
    if len(ys) < 2:
        continue
    first, last = mem[ys[0]], mem[ys[-1]]
    persist = len(first & last) / len(first) if first else np.nan
    onestep = np.mean([len(mem[a] & mem[b]) / len(mem[a])
                       for a, b in zip(ys, ys[1:]) if mem[a]])
    ever = set().union(*mem.values())
    always = set.intersection(*mem.values())
    rows.append({"group": lab, "size_2016": len(first), "size_2025": len(last),
                 "in_both_endpoints": len(first & last),
                 "persistence_endpoint": round(persist, 3),
                 "mean_year_to_year_retention": round(float(onestep), 3),
                 "ever_in": len(ever), "in_every_year": len(always)})
pers = pd.DataFrame(rows)
display(pers)
savetable(pers, "h4_tail_persistence")

top1_always = sorted(set.intersection(*[
    set(d.loc[(d.year == y) & (d.cap_share >= d[d.year == y].cap_share.quantile(0.99)),
              "ticker"]) for y in PAPER_YEARS]))
print(f"\nfirms in the top 1% in EVERY year: {top1_always}")
print("\nH4 is not falsified by a stable tail -- ten years is short. But a tail")
print("that has not moved at all sits awkwardly beside the historical record,")
print("and the paper should report the number rather than assert the analogy.")

## 10 · Figures and download

In [ ]:
# =============================================================================
# CELL 10 - Figures, manifest, zip
# =============================================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
INK, ACC, GREY = "#1f3b57", "#c8702a", "#7a7a7a"
plt.rcParams.update({"font.family": "serif", "font.size": 9, "figure.dpi": 150,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})

def savefig(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(FIGS / f"{name}.{ext}", bbox_inches="tight")
    plt.close(fig); print(f"  saved figure {name}")

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0))
ax = axes[0]
piv = F.pivot_table(index="year", columns="group3", values="asset_light")
for col, c in zip(["tail", "digital", "rest"], [INK, ACC, GREY]):
    if col in piv.columns:
        ax.plot(piv.index, piv[col], "o-", color=c, lw=2, ms=4, label=col)
ax.set_xlabel("year"); ax.set_ylabel("1 - PP&E / assets")
ax.set_title("Is the top of the index still asset-light?"); ax.legend(fontsize=8)

ax = axes[1]
pc = F.pivot_table(index="year", columns="group3", values="capex_intensity")
for col, c in zip(["tail", "digital", "rest"], [INK, ACC, GREY]):
    if col in pc.columns:
        ax.plot(pc.index, pc[col], "o-", color=c, lw=2, ms=4, label=col)
ax.set_xlabel("year"); ax.set_ylabel("capex / revenue")
ax.set_title("The compute build-out, measured directly"); ax.legend(fontsize=8)

ax = axes[2]
sub = h1[h1.spec == "BI_B"]
xs = np.arange(len(sub))
ax.errorbar(xs, sub.coef, yerr=1.96 * sub.se, fmt="o", color=INK, capsize=4)
ax.axhline(0, color=GREY, lw=1, ls=":")
ax.set_xticks(xs); ax.set_xticklabels(sub.top_pct)
ax.set_xlabel("membership of the top k% of index value")
ax.set_ylabel("coefficient on bit intensity")
ax.set_title("Bit intensity sorts the upper half, not the tail")
fig.suptitle("Notebook 04: the mechanism", fontsize=10, color=GREY, y=1.02)
savefig(fig, "M1_mechanism")

files = sorted(p for p in OUTD.rglob("*") if p.is_file())
rows = []
for p in files:
    row = {"file": str(p.relative_to(OUTD)), "kb": round(p.stat().st_size / 1024, 1)}
    if p.suffix == ".parquet":
        try:
            row["rows"] = len(pd.read_parquet(p))
        except Exception:
            row["rows"] = None
    rows.append(row)
manifest = pd.DataFrame(rows); manifest.to_csv(OUTD / "MANIFEST.csv", index=False)
display(manifest)
(OUTD / "run_summary.json").write_text(json.dumps(
    {"run": RUN_TAG, "when": datetime.datetime.now().isoformat(timespec="seconds"),
     "tables": saved_tables}, indent=2, default=str))

print("\n" + "=" * 70)
print("SEND ME")
print("=" * 70)
print("  trend_asset_light_by_group.csv + trend_capex_intensity_by_group.csv")
print("      -- whether the top of the index is becoming capital-intensive")
print("  h1_tail_membership.csv + h1_gabaix_ibragimov.csv     H1, properly tested")
print("  h3_gibrat.csv + h3_growth_variance_by_size.csv       H3")
print("  h4_tail_persistence.csv                              H4")
print("  and the firm-by-firm asset-lightness and capex tables from cell 4,")
print("  which decide whether the capital story is general or is Microsoft.")

stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
archive = shutil.make_archive(str(BASE / f"economy_of_bits_{RUN_TAG}_{stamp}"),
                              "zip", root_dir=OUTD.parent, base_dir=OUTD.name)
print(f"\nzipped -> {archive}  ({pathlib.Path(archive).stat().st_size/1024:.0f} KB)")
try:
    from google.colab import files as colab_files
    colab_files.download(archive); print("Download started.")
except ImportError:
    print("Not on Colab. Copy the archive from the path above.")